In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from tqdm import tqdm
from typing import Tuple, Optional

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from typing import Callable, Iterable, List, Tuple, Dict, Any
from math import floor

# Load Dataset

# Breast Cancer Dataset

In [ ]:
def load_binary_dataset():
    data = load_breast_cancer()
    Z = data.data
    y = data.target

    # Convert labels from {0,1} to {-1,+1} for SVM
    y = np.where(y == 0, -1, 1)

    # Train/test split
    Z_train, Z_test, y_train, y_test = train_test_split(
        Z, y, test_size=0.2, random_state=42, shuffle=True
    )

    return Z_train, y_train, Z_test, y_test


# Example usage
Z_train, y_train, Z_test, y_test = load_binary_dataset()
print("Train shape:", Z_train.shape)
print("Test shape:", Z_test.shape)

# Code to Calculate Test data Misclassifiocation Error

In [ ]:
def calc_test_error(w: np.ndarray,
                    b: float,
                    Z_test: np.ndarray,
                    y_test: np.ndarray) -> float:
    """
    Compute test misclassification error using rule:

        s_i = y_i * (w^T z_i + b)

    If s_i < 0  → misclassified
    If s_i > 0  → correct
    If s_i = 0  → misclassified with probability 0.5
    """

    # Compute signed margin s_i
    scores = y_test * (Z_test.dot(w) + b)

    # Case s_i < 0: definitely misclassified
    miscl = np.sum(scores < 0)

    # Case s_i = 0: misclassified with probability 0.5
    zero_mask = (scores == 0)
    num_zero = np.sum(zero_mask)

    if num_zero > 0:
        # Bernoulli(0.5) for ties
        miscl += np.sum(np.random.rand(num_zero) < 0.5)

    # Normalize
    return miscl / len(y_test)


# Initialization of w, b and $\xi$, and Number of code runs

In [ ]:
no_of_exps = 5
Total_itr = 2000

# Number of features and samples
d = Z_train.shape[1]   # feature dimension
m = Z_train.shape[0]   # number of training samples

# Initialize weights, bias, and slack
w_init = np.zeros(d)        # start with zero vector
b_init = 0.0                # start bias at zero
xi_init = np.ones(m) * 1.0  # slack start at 1 (or any positive value)

# Initialization of $\beta$, regulaizer $C$ and $r_0$. We also define N_schedule $N_k = \lceil k^{1/2} \rceil$.

In [ ]:
beta = 1
C = 1e-6
r0 = 1e-2

# Sample schedule for randomized feasibility
def N_schedule(k):
    #return int(np.ceil((k+1)**(0.5)))
    return 2000

# Randomized feasibility code

In [ ]:
def randomized_feasibility_update(
    N_k: int,
    w_bar: np.ndarray,
    b_bar: float,
    xi_bar: np.ndarray,
    Z: np.ndarray,
    y: np.ndarray,
    beta: float
    #eps: float = 1e-12
) -> Tuple[np.ndarray, float, np.ndarray]:
    """
    Perform N_k randomized feasibility updates starting from (w_bar, b_bar, xi_bar).

    Parameters
    ----------
    N_k : int
        Number of randomized update iterations.
    w_bar : (d,) ndarray
        Initial weight vector.
    b_bar : float
        Initial bias.
    xi_bar : (m,) ndarray
        Initial slack vector.
    Z : (m, d) ndarray train dataset
        Data matrix, each row is z_j.
    y : (m,) ndarray train labels
        Labels (+1 / -1) for each sample.
    beta : float
        Step size multiplier in gamma calculation.


    Returns
    -------
    z_w : (d,) ndarray
        Updated weight vector.
    z_b : float
        Updated bias.
    z_xi : (m,) ndarray
        Updated slack vector.
    """

    m, d = Z.shape
    z_w = w_bar.copy()
    z_b = float(b_bar)
    z_xi = xi_bar.copy()

    for _ in range(N_k):
        j = np.random.randint(0, m)       # sample random index in {0,...,m-1}
        z_j = Z[j]                        # shape (d,). ----> Check later on if train data is given row wise or column wise
        y_j = float(y[j])

        # Compute margin residual: (1 - xi_j - y_j * (z_w^T z_j + z_b))
        margin_resid = 1.0 - z_xi[j] - y_j * (z_w.dot(z_j) + z_b)

        # Denominator: ||y_j z_j||^2 + y_j^2 + 1
        denom = (y_j ** 2) * (z_j.dot(z_j)) + (y_j ** 2) + 1.0
        #denom = float(denom) + eps

        gamma = beta * (margin_resid) / denom

        # Update rules (only entry j of xi changes)
        z_w += gamma * y_j * z_j
        z_b += gamma * y_j
        # Only j-th slack variable updated and then projected to >= 0
        z_xi[j] = max(0.0, z_xi[j] + gamma)

    return z_w, z_b, z_xi

In [ ]:

def svm_dows_step(
    w_init: np.ndarray,
    b_init: float,
    xi_init: np.ndarray,
    Z: np.ndarray,
    y: np.ndarray,
    C: float,
    N_schedule,
    beta: float,
    T: int,
    r0: float               # initial radius estimate
):
    """
    One outer iteration that forms (w_bar, b_bar, xi_bar) and calls the randomized feasibility update.

    Parameters
    ----------
    w_init : (d,) ndarray
        Current weight vector.
    b_init : float
        Current bias.
    xi_init : (m,) ndarray
        Current slack variables.
    Z : (m, d) ndarray
        Data matrix.
    y : (m,) ndarray
        Labels for data.
    C : float
        Constant regularizer used in xi.
    N_schedule : callable(k)
        Number of samples N_k for inner iterations.
    beta : float
        Step-size multiplier used in randomized feasibility update.
    T : int
        Total number of outer iterations.
    r0 : float
        Initial radius estimate.

    Returns
    -------
    obj_list, max_viol_list, total_viol_list, test_miscl_err_frac.
    """

    # Dimensions
    m, d = Z.shape
    #assert xi0.shape[0] == m
    # initialize
    w = w_init.copy().astype(float)
    b = float(b_init)
    xi = xi_init.copy().astype(float)

    # x0 (for radius calc) = initial point concatenated
    x0 = np.concatenate([w, np.array([b]), xi])
    x_k = x0.copy()

    # bookkeeping for adaptive steps
    r_prev = float(r0)
    p_prev = 0.0

    # accumulators for weighted mean (user sketch)
    num = np.zeros_like(x0, dtype=float)
    den = 0.0


    obj_list = []
    max_viol_list = []
    total_viol_list = []
    test_miscl_err_frac = []

    for k in tqdm(range(T), desc="svm_dows outer iters"):

        # compute subgradient s_f_k = [w, 0, C * ones]
        s_f_k = np.concatenate([w, np.array([0.0]), C * np.ones_like(xi)])

        # radius
        # note: x_k currently corresponds to [w, b, xi] (from previous iter)
        r_k = max(np.linalg.norm(x_k - x0), r_prev)

        # update p_k
        norm_s_sq = np.linalg.norm(s_f_k) ** 2
        p_k = p_prev + (r_k ** 2) * norm_s_sq

        # compute alpha_k per your formula (guard sqrt)
        sqrt_p_k = np.sqrt(p_k)
        alpha_k = (r_k ** 2) / sqrt_p_k

        # Form w_bar, b_bar, xi_bar according to pseudo-code
        w_bar = (1.0 - alpha_k) * w
        b_bar = float(b)
        xi_bar = np.maximum(0.0, xi - alpha_k * C * np.ones_like(xi))

        N_k = N_schedule(k)
        # Call randomized feasibility update
        w_new, b_new, xi_new = randomized_feasibility_update(
            N_k=N_k,
            w_bar=w_bar,
            b_bar=b_bar,
            xi_bar=xi_bar,
            Z=Z,
            y=y,
            beta=beta
        )

        # update x_k and bookkeeping
        x_k = np.concatenate([w_new, np.array([b_new]), xi_new])

        # update accumulators (choose weight = r_k**2 here)
        weight_k = r_k ** 2
        num += weight_k * x_k
        den += weight_k
        bar_x_k = num / den

        # Extract components
        bar_w  = bar_x_k[:d]           # first d entries
        bar_b  = bar_x_k[d]            # the (d)-th entry (scalar)
        bar_xi = bar_x_k[d+1 : d+1+m]  # remaining m entries

        # compute objective and constraint violation for logging
        obj_list.append(0.5 * np.linalg.norm(bar_w) ** 2 + C * np.sum(bar_xi))
        # constraint violations: max(0, 1 - avg_xi_i - y_i (avg_w^T z_i + avg_b))
        margins = 1.0 - bar_xi - y * (Z.dot(bar_w) + bar_b)
        max_viol_list.append(float(np.max(np.maximum(0.0, margins))))
        total_viol_list.append(float(np.sum(np.maximum(0.0, margins))))

        # prepare for next iter
        w = w_new
        b = b_new
        xi = xi_new
        r_prev = r_k
        p_prev = p_k

        ## Add code for calculating the test error
        test_miscl_err_frac.append(calc_test_error(bar_w, bar_b, Z_test, y_test))

    return obj_list, max_viol_list, total_viol_list, test_miscl_err_frac

# Run multiple experiments for DoWS

In [ ]:
def run_multiple_experiments(
    n_experiments: int,
    svm_dows_step_fn,
    T: int,
    w_init, b_init, xi_init,
    Z, y,
    Z_test, y_test,
    C, N_schedule, beta, r0
):
    """
    Runs svm_dows_step for multiple experiments and stores all outputs.

    Returns:
        obj_all:            array shape (n_experiments, T)
        max_viol_all:       array shape (n_experiments, T)
        total_viol_all:     array shape (n_experiments, T)
        test_err_all:       array shape (n_experiments, T)
    """

    obj_all = np.zeros((n_experiments, T))
    max_viol_all = np.zeros((n_experiments, T))
    total_viol_all = np.zeros((n_experiments, T))
    test_err_all = np.zeros((n_experiments, T))

    for e in range(n_experiments):
        print(f"Running experiment {e+1}/{n_experiments}...")

        obj_list, max_viol_list, total_viol_list, test_miscl_err_frac = svm_dows_step_fn(
            w_init=w_init.copy(),
            b_init=b_init,
            xi_init=xi_init.copy(),
            Z=Z,
            y=y,
            C=C,
            N_schedule=N_schedule,
            beta=beta,
            T=T,
            r0=r0
        )

        obj_all[e, :] = np.array(obj_list)
        max_viol_all[e, :] = np.array(max_viol_list)
        total_viol_all[e, :] = np.array(total_viol_list)
        test_err_all[e, :] = np.array(test_miscl_err_frac)

    return obj_all, max_viol_all, total_viol_all, test_err_all


In [ ]:


obj_all_dows, max_viol_all_dows, total_viol_all_dows, test_err_all_dows = run_multiple_experiments(
    n_experiments=no_of_exps,
    svm_dows_step_fn=svm_dows_step,
    T=Total_itr,
    w_init=w_init,
    b_init=b_init,
    xi_init=xi_init,
    Z=Z_train,
    y=y_train,
    Z_test=Z_test,
    y_test=y_test,
    C=C,
    N_schedule=N_schedule,
    beta=beta,
    r0=r0
)


# Run T-DOWS

In [ ]:

def svm_tdows_step(
    w_init: np.ndarray,
    b_init: float,
    xi_init: np.ndarray,
    Z: np.ndarray,
    y: np.ndarray,
    C: float,
    N_schedule,
    beta: float,
    T: int,
    r0: float               # initial radius estimate
):
    """
    One outer iteration that forms (w_bar, b_bar, xi_bar) and calls the randomized feasibility update.

    Parameters
    ----------
    w_init : (d,) ndarray
        Current weight vector.
    b_init : float
        Current bias.
    xi_init : (m,) ndarray
        Current slack variables.
    Z : (m, d) ndarray
        Data matrix.
    y : (m,) ndarray
        Labels for data.
    C : float
        Constant regularizer used in xi.
    N_schedule : callable(k)
        Number of samples N_k for inner iterations.
    beta : float
        Step-size multiplier used in randomized feasibility update.
    T : int
        Total number of outer iterations.
    r0 : float
        Initial radius estimate.

    Returns
    -------
    obj_list, max_viol_list, total_viol_list, test_miscl_err_frac.
    """

    # Dimensions
    m, d = Z.shape
    #assert xi0.shape[0] == m
    # initialize
    w = w_init.copy().astype(float)
    b = float(b_init)
    xi = xi_init.copy().astype(float)

    # x0 (for radius calc) = initial point concatenated
    x0 = np.concatenate([w, np.array([b]), xi])
    x_k = x0.copy()

    # bookkeeping for adaptive steps
    r_prev = float(r0)
    p_prev = 0.0

    # accumulators for weighted mean (user sketch)
    num = np.zeros_like(x0, dtype=float)
    den = 0.0


    obj_list = []
    max_viol_list = []
    total_viol_list = []
    test_miscl_err_frac = []

    for k in tqdm(range(T), desc="svm_tdows outer iters"):

        # compute subgradient s_f_k = [w, 0, C * ones]
        s_f_k = np.concatenate([w, np.array([0.0]), C * np.ones_like(xi)])

        # radius
        # note: x_k currently corresponds to [w, b, xi] (from previous iter)
        r_k = max(np.linalg.norm(x_k - x0), r_prev)

        # update p_k
        norm_s_sq = np.linalg.norm(s_f_k) ** 2
        p_k = p_prev + (r_k ** 2) * norm_s_sq

        # compute alpha_k per formula
        if k == 0:
            p_1 = p_k

        alpha_k = r_k**2 / (np.sqrt(2*p_k)*np.log((np.e)*p_k/p_1))

        # Form w_bar, b_bar, xi_bar according to pseudo-code
        w_bar = (1.0 - alpha_k) * w
        b_bar = float(b)
        xi_bar = np.maximum(0.0, xi - alpha_k * C * np.ones_like(xi))

        N_k = N_schedule(k)
        # Call randomized feasibility update
        w_new, b_new, xi_new = randomized_feasibility_update(
            N_k=N_k,
            w_bar=w_bar,
            b_bar=b_bar,
            xi_bar=xi_bar,
            Z=Z,
            y=y,
            beta=beta
        )

        # update x_k and bookkeeping
        x_k = np.concatenate([w_new, np.array([b_new]), xi_new])

        # update accumulators (choose weight = r_k**2 here)
        weight_k = r_k ** 2
        num += weight_k * x_k
        den += weight_k
        bar_x_k = num / den

        # Extract components
        bar_w  = bar_x_k[:d]           # first d entries
        bar_b  = bar_x_k[d]            # the (d)-th entry (scalar)
        bar_xi = bar_x_k[d+1 : d+1+m]  # remaining m entries

        # compute objective and constraint violation for logging
        obj_list.append(0.5 * np.linalg.norm(bar_w) ** 2 + C * np.sum(bar_xi))
        # constraint violations: max(0, 1 - avg_xi_i - y_i (avg_w^T z_i + avg_b))
        margins = 1.0 - bar_xi - y * (Z.dot(bar_w) + bar_b)
        max_viol_list.append(float(np.max(np.maximum(0.0, margins))))
        total_viol_list.append(float(np.sum(np.maximum(0.0, margins))))

        # prepare for next iter
        w = w_new
        b = b_new
        xi = xi_new
        r_prev = r_k
        p_prev = p_k

        ## Add code for calculating the test error
        test_miscl_err_frac.append(calc_test_error(bar_w, bar_b, Z_test, y_test))

    return obj_list, max_viol_list, total_viol_list, test_miscl_err_frac

In [ ]:
obj_all_tdows, max_viol_all_tdows, total_viol_all_tdows, test_err_all_tdows = run_multiple_experiments(
    n_experiments=no_of_exps,
    svm_dows_step_fn=svm_dows_step,
    T=Total_itr,
    w_init=w_init,
    b_init=b_init,
    xi_init=xi_init,
    Z=Z_train,
    y=y_train,
    Z_test=Z_test,
    y_test=y_test,
    C=C,
    N_schedule=N_schedule,
    beta=beta,
    r0=r0
)


# Lagrangian Primal-Dual

In [ ]:
def arrow_hurwicz(
    w_init: np.ndarray,
    b_init: float,
    xi_init: np.ndarray,
    primal_step_schedule,
    dual_step_schedule,
    Z: np.ndarray,
    y: np.ndarray,
    C: float,
    T: int
):

    """
    The method uses a single data point in each iteration.

    Parameters
    ----------
    w_init : (d,) ndarray
        Current weight vector.
    b_init : float
        Current bias.
    xi_init : (m,) ndarray
        Current slack variables.
    primal_step_schedule: callable(k)
        Step-size for primal update.
    dual_step_schedule: callable(k)
        Step-size for dual update.
    Z : (m, d) ndarray
        Train Data matrix.
    y : (m,) ndarray
        Train Labels for data.
    C : float
        Constant regularizer used in xi.
    T : int
        Total number of outer iterations.

    Returns
    -------
    obj_list, max_viol_list, total_viol_list, test_miscl_err_frac.
    """

    # Dimensions
    m, d = Z.shape
    #assert xi0.shape[0] == m
    # initialize
    w = w_init.copy().astype(float)
    b = float(b_init)
    xi = xi_init.copy().astype(float)

    lam = np.zeros(m, dtype=float)   # dual vars (lambda), in [0, C]

    obj_list = []
    max_viol_list = []
    total_viol_list = []
    test_miscl_err_frac = []

    for it in tqdm(range(T), desc="primal-dual outer iters"):
        # fetch step sizes from user-provided schedules (must be defined)
        try:
            eta = primal_step_schedule(it)   # primal step
        except NameError:
            # fallback default
            eta = 1e-2
        try:
            tau = dual_step_schedule(it)     # dual step
        except NameError:
            # fallback default
            tau = 1e-2

        # ---------- Primal descent step ----------
        # target_w = sum_i lam_i * y_i * z_i  (vectorized)
        target_w = (lam * y) @ Z    # shape (d,)

        # gradient for w: w - target_w  => descent step
        w = w - eta * (w - target_w)

        # gradient for b: - sum_i lam_i y_i  => descent step on L wrt b
        b = b + eta * np.sum(lam * y)

        # primal slack xi update: gradient is (C - lam) (from stationarity)
        xi = xi - eta * (C - lam)
        # projection xi >= 0
        np.maximum(xi, 0.0, out=xi)

        # ---------- Dual ascent step ----------
        # compute current margins based on updated primal
        margins = y * (Z.dot(w) + b)   # length m

        # dual gradient g = 1 - xi - margins
        g_lam = 1.0 - xi - margins

        # ascent
        lam = lam + tau * g_lam
        #np.maximum(lam, 0.0, out=lam)
        np.clip(lam, 0.0, C, out=lam)

        # ---------- Logging: objective & violations ----------
        obj = 0.5 * np.linalg.norm(w) ** 2 + C * np.sum(xi)
        resid = 1.0 - xi - y * (Z.dot(w) + b)
        max_viol = float(np.max(np.maximum(0.0, resid)))
        total_viol = float(np.sum(np.maximum(0.0, resid)))

        obj_list.append(obj)
        max_viol_list.append(max_viol)
        total_viol_list.append(total_viol)

        # compute test error if test set available
        try:
            test_err = calc_test_error(w, b, Z_test, y_test)
        except NameError:
            test_err = np.nan
        test_miscl_err_frac.append(test_err)

    return obj_list, max_viol_list, total_viol_list, test_miscl_err_frac

In [ ]:
def run_multiple_experiments_pd(
    n_experiments: int,
    arrow_hurwicz,
    T: int,
    w_init: np.ndarray,
    b_init: float,
    xi_init: np.ndarray,
    primal_step_schedule,
    dual_step_schedule,
    Z: np.ndarray,
    y: np.ndarray,
    C: float
):
    """

    Parameters
    ----------
    n_experiments : int
        Number of independent experiments to run.
    arrow_hurwicz: primal dual func
    T : int
        Number of iterations produced by arrow_hurwicz.
    w_init, b_init, xi_init :
        Initial primal variables (will be copied for each experiment).
    primal_step_schedule : callable(k) -> float
    dual_step_schedule : callable(k) -> float
    Z, y : training data
    C : float
        Regularization parameter.

    Returns
    -------
    obj_all, max_viol_all, total_viol_all, test_err_all : np.ndarray
        Each has shape (n_experiments, T).
    """
    # Preallocate result arrays
    obj_all = np.zeros((n_experiments, T), dtype=float)
    max_viol_all = np.zeros((n_experiments, T), dtype=float)
    total_viol_all = np.zeros((n_experiments, T), dtype=float)
    test_err_all = np.zeros((n_experiments, T), dtype=float)

    for e in range(n_experiments):
        print(f"Experiment {e+1}/{n_experiments} ...")

        # Fresh copies for this experiment
        w0 = w_init.copy()
        b0 = float(b_init)
        xi0 = xi_init.copy()

        # Call solver
        obj_list, max_viol_list, total_viol_list, test_miscl_err_frac = arrow_hurwicz(
            w_init=w0,
            b_init=b0,
            xi_init=xi0,
            primal_step_schedule=primal_step_schedule,
            dual_step_schedule=dual_step_schedule,
            Z=Z,
            y=y,
            C=C,
            T=T
        )

        # Convert returned lists to arrays and store
        if len(obj_list) != T:
            raise ValueError(f"solver_fn returned obj_list of length {len(obj_list)} but expected T={T}")
        if len(total_viol_list) != T:
            raise ValueError(f"solver_fn returned total_viol_list of length {len(total_viol_list)} but expected T={T}")
        if len(test_miscl_err_frac) != T:
            raise ValueError(f"solver_fn returned test_miscl_err_frac of length {len(test_miscl_err_frac)} but expected T={T}")

        obj_all[e, :] = np.asarray(obj_list, dtype=float)
        max_viol_all[e, :] = np.asarray(max_viol_list, dtype=float)
        total_viol_all[e, :] = np.asarray(total_viol_list, dtype=float)
        test_err_all[e, :] = np.asarray(test_miscl_err_frac, dtype=float)

    return obj_all, max_viol_all, total_viol_all, test_err_all

# Cross-Validation to tune the step-sizes

In [ ]:
def _ensure_schedule(obj):
    """Convert scalar to constant schedule, keep callables unchanged."""
    if callable(obj):
        return obj
    val = float(obj)
    return lambda k, val=val: val

def _make_folds(n: int, k_folds: int, rng):
    """Return list of (train_idx, val_idx) for k-fold CV."""
    idx = np.arange(n)
    rng.shuffle(idx)
    fold_sizes = [(n // k_folds) + (1 if i < (n % k_folds) else 0)
                  for i in range(k_folds)]
    folds = []
    start = 0
    for size in fold_sizes:
        end = start + size
        val = idx[start:end]
        train = np.setdiff1d(idx, val, assume_unique=True)
        folds.append((train, val))
        start = end
    return folds

def cross_validate_step_schedules(
    arrow_hurwicz_fn: Callable[..., Tuple[list, list, list, list]],
    Z: np.ndarray,
    y: np.ndarray,
    C: float,
    primal_candidates: Iterable,
    dual_candidates: Iterable,
    T_cv: int,
    k_folds: int = 3,
    val_fraction: float = None
) -> Dict[str, Any]:
    """
    Cross-validation for selecting primal_step_schedule and dual_step_schedule,
    WITHOUT using any fixed random seed.
    """

    rng = np.random.default_rng()  # <-- no fixed seed
    n = Z.shape[0]

    # Convert schedules
    primal_scheds = [_ensure_schedule(p) for p in primal_candidates]
    dual_scheds = [_ensure_schedule(q) for q in dual_candidates]

    # Build folds or a single train/validation split
    if val_fraction is not None:
        idx = np.arange(n)
        rng.shuffle(idx)
        n_val = int(floor(val_fraction * n))
        val_idx = idx[:n_val]
        train_idx = idx[n_val:]
        folds = [(train_idx, val_idx)]
    else:
        folds = _make_folds(n, k_folds, rng)

    grid_results = []

    # Temporary override of global test data for validation evaluation
    globs = globals()
    saved_Z_test = globs.get("Z_test", None)
    saved_y_test = globs.get("y_test", None)

    for i_p, p_sched in enumerate(primal_scheds):
        for i_d, d_sched in enumerate(dual_scheds):
            fold_errs = []

            for (train_idx, val_idx) in folds:
                # Extract train and validation sets
                Z_train = Z[train_idx]
                y_train = y[train_idx]
                Z_val = Z[val_idx]
                y_val = y[val_idx]

                # Inject validation set so arrow_hurwicz can evaluate test error
                globs["Z_test"] = Z_val
                globs["y_test"] = y_val

                # Initialize primal variables
                d = Z_train.shape[1]
                w0 = np.zeros(d)
                b0 = 0.0
                xi0 = np.ones(len(train_idx))

                # Run the solver
                _, _, _, test_miscl_err_frac = arrow_hurwicz_fn(
                    w_init=w0,
                    b_init=b0,
                    xi_init=xi0,
                    primal_step_schedule=p_sched,
                    dual_step_schedule=d_sched,
                    Z=Z_train,
                    y=y_train,
                    C=C,
                    T=T_cv
                )

                # Use the final validation error for this fold
                val_err = float(test_miscl_err_frac[-1])
                fold_errs.append(val_err)

            grid_results.append({
                "primal_schedule": p_sched,
                "dual_schedule": d_sched,
                "primal": primal_candidates[i_p],
                "dual": dual_candidates[i_d],
                "per_fold_errs": fold_errs,
                "mean_val_err": float(np.mean(fold_errs)),
                "std_val_err": float(np.std(fold_errs))
            })

    # Restore global values
    if saved_Z_test is None:
        globs.pop("Z_test", None)
    else:
        globs["Z_test"] = saved_Z_test

    if saved_y_test is None:
        globs.pop("y_test", None)
    else:
        globs["y_test"] = saved_y_test

    # Choose best schedule pair
    best_entry = min(grid_results, key=lambda e: e["mean_val_err"])

    return {
        "best_primal_schedule": best_entry["primal_schedule"],
        "best_dual_schedule": best_entry["dual_schedule"],
        "grid_results": grid_results,
        "primal_candidates": primal_candidates,
        "dual_candidates": dual_candidates
    }


In [ ]:
cv_out = cross_validate_step_schedules(
    arrow_hurwicz_fn = arrow_hurwicz,
    Z = Z_train,
    y = y_train,
    C = C,
    primal_candidates = [1e-4, 1e-5, 1e-6],
    dual_candidates   = [1e-4, 1e-5, 1e-6],
    T_cv = 200,
    k_folds = 3
)

best_primal = cv_out["best_primal_schedule"]
best_dual   = cv_out["best_dual_schedule"]

print(f"Best primal schedule: {best_primal}")
print(f"Best dual schedule: {best_dual}")


In [ ]:
best_primal(0)

# Run multiple experiment of primal-dual with best step size

In [ ]:
obj_all_pd, max_viol_all_pd, total_viol_all_pd, test_err_all_pd = run_multiple_experiments_pd(
    n_experiments=no_of_exps,
    arrow_hurwicz=arrow_hurwicz,
    T=Total_itr,
    w_init=w_init,
    b_init=b_init,
    xi_init=xi_init,
    primal_step_schedule=best_primal,
    dual_step_schedule=best_dual,
    Z=Z_train,
    y=y_train,
    C=C
)

In [ ]:
# L_obj = 1

# def primal_step_schedule(k):
#     #return 1 / (2* L_obj)   # or whatever schedule you prefer
#     #return 1 / np.sqrt(Total_itr)
#     return 1e-2

# def dual_step_schedule(k):
#     #return 1 / (2* L_obj)  # or any schedule you want
#     #return 1 / np.sqrt(Total_itr)
#     return 1e-2


# obj_all_pd, max_viol_all_pd, total_viol_all_pd, test_err_all_pd = run_multiple_experiments_pd(
#     n_experiments=no_of_exps,
#     arrow_hurwicz=arrow_hurwicz,
#     T=Total_itr,
#     w_init=w_init,
#     b_init=b_init,
#     xi_init=xi_init,
#     primal_step_schedule=primal_step_schedule,
#     dual_step_schedule=dual_step_schedule,
#     Z=Z_train,
#     y=y_train,
#     C=C
# )

# Compute the mean and std for the outputs across experiemnts

In [ ]:
def compute_mean_std(
    obj: np.ndarray,
    total_viol: np.ndarray,
    test_err: np.ndarray
):
    """
    Computes mean and std for objective, total violation, and test error.

    Parameters
    ----------
    obj_all           : array (n_experiments, T)
    total_viol_all    : array (n_experiments, T)
    test_err_all      : array (n_experiments, T)

    Returns:
        obj_mean, obj_std, viol_mean, viol_std, err_mean, err_std
    """

    # Compute stats
    obj_mean, obj_std = obj.mean(axis=0), obj.std(axis=0)
    viol_mean, viol_std = total_viol.mean(axis=0), total_viol.std(axis=0)
    err_mean, err_std = test_err.mean(axis=0), test_err.std(axis=0)

    return obj_mean, obj_std, viol_mean, viol_std, err_mean, err_std

In [ ]:
# For DoWS
(obj_mean_dows, obj_std_dows,
 viol_mean_dows, viol_std_dows,
 err_mean_dows, err_std_dows) = compute_mean_std(
     obj_all_dows, total_viol_all_dows, test_err_all_dows
 )

# For T-DoWS
(obj_mean_tdows, obj_std_tdows,
 viol_mean_tdows, viol_std_tdows,
 err_mean_tdows, err_std_tdows) = compute_mean_std(
     obj_all_tdows, total_viol_all_tdows, test_err_all_tdows
 )

# For Primal-Dual
(obj_mean_pd, obj_std_pd,
 viol_mean_pd, viol_std_pd,
 err_mean_pd, err_std_pd) = compute_mean_std(
     obj_all_pd, total_viol_all_pd, test_err_all_pd
 )

# Code for plotting the figures

In [ ]:
iters = np.arange(1, Total_itr + 1)

figure(1)
plt.plot(iters, obj_mean_dows, color='#0072B2', label='Algorithm 3')
plt.fill_between(iters, obj_mean_dows - obj_std_dows,
                 obj_mean_dows + obj_std_dows, color='#0072B2', alpha=0.2)


plt.plot(iters, obj_mean_tdows, color='#009E73', label='Algorithm 4')
plt.fill_between(iters, obj_mean_tdows - obj_std_tdows,
                 obj_mean_tdows + obj_std_tdows, color='#009E73', alpha=0.2)

plt.plot(iters, obj_mean_pd, color='#E69F00', label='Primal-Dual (with cross-validation)')
plt.fill_between(iters, obj_mean_pd - obj_std_pd,
                 obj_mean_pd + obj_std_pd, color='#E69F00', alpha=0.2)


plt.xlabel('Number of iterations', fontsize=18)
plt.ylabel('Objective function', fontsize=18)
plt.title('Breast Cancer Objective Function', fontsize=18)
plt.yscale('log')
#plt.ylim(-0.4, -0.25)
plt.grid(True)
plt.legend(fontsize=12, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
figure(2)
plt.plot(iters, viol_mean_dows, color='#0072B2', label='Algorithm 3')
plt.fill_between(iters, viol_mean_dows - viol_std_dows,
                 viol_mean_dows + viol_std_dows, color='#0072B2', alpha=0.2)

plt.plot(iters, viol_mean_tdows, color='#009E73', label='Algorithm 4')
plt.fill_between(iters, viol_mean_tdows - viol_std_tdows,
                 viol_mean_tdows + viol_std_tdows, color='#009E73', alpha=0.2)

plt.plot(iters, viol_mean_pd, color='#E69F00', label='Primal-Dual (with cross-validation)')
plt.fill_between(iters, viol_mean_pd - viol_std_pd,
                 viol_mean_pd + viol_std_pd, color='#E69F00', alpha=0.2)

plt.xlabel('Number of iterations', fontsize=18)
plt.ylabel(r'$\sum_{i=1}^m g^{+}_i(w,b,\xi_i)$ over train data', fontsize=18)
plt.title('Breast Cancer Infeasibility', fontsize=18)
plt.yscale('log')
#plt.ylim(-0.4, -0.25)
plt.grid(True)
plt.legend(fontsize=12, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
figure(3)
plt.plot(iters, err_mean_dows, color='#0072B2', label='Algorithm 3')
plt.fill_between(iters, err_mean_dows - err_std_dows,
                 err_mean_dows + err_std_dows, color='#0072B2', alpha=0.2)

plt.plot(iters, err_mean_tdows, color='#009E73', label='Algorithm 4')
plt.fill_between(iters, err_mean_tdows - err_std_tdows,
                 err_mean_tdows + err_std_tdows, color='#009E73', alpha=0.2)

plt.plot(iters, err_mean_pd, color='#E69F00', label='Primal-Dual (with cross-validation)')
plt.fill_between(iters, err_mean_pd - err_std_pd,
                 err_mean_pd + err_std_pd, color='#E69F00', alpha=0.2)

plt.xlabel('Number of iterations', fontsize=18)
plt.ylabel('Test Misclassification Error', fontsize=18)
plt.title('Breast Cancer Test Error', fontsize=18)
plt.yscale('log')
#plt.ylim(0.07, 0.7)
plt.grid(True)
plt.legend(fontsize=12, loc='best')
plt.tight_layout()
plt.show()